# 📦 Notebook 01 — Data Creation & Exploration

This notebook explores the raw data assets that power SmartCart AI:
- `menu_items.json` — 110 Indian food items with KPT, margin, price, category
- `food_pairings.json` — 159 curated food pairings with strength scores
- `restaurant_profiles.json` — 22 restaurant profiles

We'll visualize distributions of categories, cuisines, prices, KPT, and pairing strengths.

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BASE = os.path.join(os.path.dirname(os.getcwd()), 'SmartCart-AI-Context-Aware-Revenue-Optimized-CSAO-Engine')
DATA_RAW = os.path.join(os.getcwd(), '..', 'data', 'raw')
print('Data path:', os.path.abspath(DATA_RAW))

In [ ]:
# Load menu items
menu_path = os.path.join(DATA_RAW, 'menu_items.json')
with open(menu_path) as f:
    menu_data = json.load(f)

# Handle both list and dict-of-lists formats
if isinstance(menu_data, list):
    df_menu = pd.DataFrame(menu_data)
elif isinstance(menu_data, dict) and 'items' in menu_data:
    df_menu = pd.DataFrame(menu_data['items'])
else:
    df_menu = pd.DataFrame(list(menu_data.values()))

print(f'Menu items loaded: {len(df_menu)} rows')
print(f'Columns: {df_menu.columns.tolist()}')
df_menu.head()

In [ ]:
# Category distribution
if 'category' in df_menu.columns:
    cat_counts = df_menu['category'].value_counts()
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(cat_counts.index, cat_counts.values, color='#FF6B35', edgecolor='white', linewidth=0.5)
    ax.set_title('Menu Items by Category', fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('No category column found. Columns:', df_menu.columns.tolist())

In [ ]:
# Price distribution
price_col = next((c for c in ['price', 'price_inr', 'base_price'] if c in df_menu.columns), None)
if price_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram
    axes[0].hist(df_menu[price_col], bins=20, color='#2ECC71', edgecolor='white', linewidth=0.5)
    axes[0].set_title('Price Distribution (₹)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Price (₹)')
    axes[0].set_ylabel('Count')
    axes[0].axvline(df_menu[price_col].mean(), color='red', linestyle='--', label=f'Mean ₹{df_menu[price_col].mean():.0f}')
    axes[0].legend()

    # Box plot by category
    if 'category' in df_menu.columns:
        cat_price = df_menu.groupby('category')[price_col].median().sort_values(ascending=False).head(8)
        axes[1].barh(cat_price.index, cat_price.values, color='#3498DB')
        axes[1].set_title('Median Price by Category (₹)', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Median Price (₹)')

    plt.tight_layout()
    plt.show()
    print(f'Price stats: min=₹{df_menu[price_col].min()}, max=₹{df_menu[price_col].max()}, mean=₹{df_menu[price_col].mean():.1f}')
else:
    print('No price column found')

In [ ]:
# KPT (Kitchen Prep Time) distribution
kpt_col = next((c for c in ['kpt_minutes', 'kpt', 'prep_time', 'prep_time_minutes'] if c in df_menu.columns), None)
if kpt_col:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df_menu[kpt_col], bins=15, color='#9B59B6', edgecolor='white', linewidth=0.5)
    ax.axvline(df_menu[kpt_col].mean(), color='orange', linestyle='--',
               label=f'Mean {df_menu[kpt_col].mean():.1f} min')
    ax.set_title('Kitchen Prep Time (KPT) Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('KPT (minutes)')
    ax.set_ylabel('Count')
    ax.legend()

    # KPT buckets
    bins_kpt = [0, 5, 10, 20, 60]
    labels = ['Instant (0-5)', 'Quick (5-10)', 'Standard (10-20)', 'Slow (20+)']
    df_menu['kpt_bucket'] = pd.cut(df_menu[kpt_col], bins=bins_kpt, labels=labels)
    print(df_menu['kpt_bucket'].value_counts())

    plt.tight_layout()
    plt.show()
else:
    print('No KPT column found. Available columns:', df_menu.columns.tolist())

In [ ]:
# Food pairings
pairings_path = os.path.join(DATA_RAW, 'food_pairings.json')
with open(pairings_path) as f:
    pairings_data = json.load(f)

if isinstance(pairings_data, list):
    df_pairs = pd.DataFrame(pairings_data)
elif isinstance(pairings_data, dict) and 'pairings' in pairings_data:
    df_pairs = pd.DataFrame(pairings_data['pairings'])
else:
    df_pairs = pd.DataFrame(list(pairings_data.values()))

print(f'Pairings loaded: {len(df_pairs)} rows')
print(df_pairs.columns.tolist())
df_pairs.head()

In [ ]:
# Pairing strength distribution
strength_col = next((c for c in ['strength', 'pairing_strength', 'score', 'weight'] if c in df_pairs.columns), None)
if strength_col:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df_pairs[strength_col], bins=20, color='#E74C3C', edgecolor='white', linewidth=0.5)
    ax.axvline(df_pairs[strength_col].mean(), color='blue', linestyle='--',
               label=f'Mean {df_pairs[strength_col].mean():.2f}')
    ax.set_title('Food Pairing Strength Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Pairing Strength Score')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Top 10 strongest pairings
    name_col = next((c for c in ['item1', 'food1', 'source', 'from'] if c in df_pairs.columns), df_pairs.columns[0])
    pair_col = next((c for c in ['item2', 'food2', 'target', 'to'] if c in df_pairs.columns), df_pairs.columns[1])
    print('\nTop 10 Strongest Pairings:')
    print(df_pairs.nlargest(10, strength_col)[[name_col, pair_col, strength_col]].to_string(index=False))
else:
    print('No strength column found. Columns:', df_pairs.columns.tolist())

In [ ]:
# Summary statistics
print('=' * 50)
print('DATA SUMMARY')
print('=' * 50)
print(f'Total menu items:    {len(df_menu)}')
print(f'Total pairings:      {len(df_pairs)}')
if 'category' in df_menu.columns:
    print(f'Unique categories:   {df_menu["category"].nunique()}')
if 'cuisine' in df_menu.columns:
    print(f'Unique cuisines:     {df_menu["cuisine"].nunique()}')
if kpt_col:
    print(f'KPT range:           {df_menu[kpt_col].min()} – {df_menu[kpt_col].max()} min')
if price_col:
    print(f'Price range:         ₹{df_menu[price_col].min()} – ₹{df_menu[price_col].max()}')